This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

## Task

We need to write a function for matching groups: a small but important function that completes the matches found by a model.

If items 1 and 2 are a match, and items 7 and 2 are a match, then it's reasonable to conclude that 1 and 7 are a match too.

In [ ]:
from typing import List
from typing import Tuple

def extend_matches(pairs: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Match predictions via transitive relation: (1, 2), (1, 3) -> (2, 3)"""
    added_pairs = []
    n = len(pairs)
    for l in range(n-1):
        p1 = set(pairs[l])
        for r in range(l+1, n):
            p2 = set(pairs[r])
            if p1 & p2:
                added_pairs.append(tuple(p1 ^ p2))
    
    pairs.extend(added_pairs)
    pairs = [sorted(p) for p in pairs]
    pairs = [p for p in pairs if p]
    pairs = [tuple(p) for p in pairs]
    pairs = list(set(pairs))
    if len(pairs) == n:
        pairs = sorted(pairs, reverse=False)
        return pairs
    else:
        pairs = extend_matches(pairs)
                
    return pairs

The solution above works, but it's inefficient.

The canonical solution is via a **Disjoint Set Union (DSU)**.

I build a `parent` dictionary that maps each value (node) to its parent node. Climbing up the chain then lets me find the root node (the one whose parent is itself).

This way I assemble a collection of disjoint subsets `groups`.

Finally, I use `itertools.combinations` to produce all possible pairs within each subset.

In [ ]:
from typing import List, Tuple, Dict
from itertools import combinations

def extend_matches(pairs: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    """Match predictions via transitive relation: (1, 2), (1, 3) -> (2, 3)"""
    parent: Dict[int, int] = {}

    def find(x: int) -> int:
        """Find a parent for the input number"""
        if x not in parent:
            parent[x] = x
        while parent[x] != x:
            x = parent[x]
        return x

    def union(x: int, y: int) -> None:
        """Unite two numbers into one subset"""
        parent[find(x)] = find(y)

    for a, b in pairs:
        union(a, b)

    #Gather all groups
    groups: Dict[int, List[int]] = {}
    for p in parent.items():
        root = find(p[1])
        groups.setdefault(root, []).append(p[0])

    #Find all combinations in each group
    all_pairs = []
    for subset in groups.values():
        all_pairs.extend(combinations(sorted(subset), 2))

    return sorted(all_pairs)


## Follow-up

Let's update the post-processing so it works not only with pairs but also with triples, quadruples, and other sets of items.

```python
from typing import List


def extend_matches(groups: List[tuple]) -> List[tuple]:
    ...
    return groups
```

The input and output format is the same — a list of tuples. The requirements are the same too: return the match groups without duplicates, everything sorted ascending (both the list itself and the items within each group).

Example input:

`[(5, 3, 4, 8), (1, 2), (7, 2)]`

Example output:

`[(1, 2, 7), (3, 4, 5, 8)]`

**Solution**

I'll build on the previous solution.

There are two possible approaches:
1. Pre-process the input — use `itertools.combinations` to turn each group into a set of pairs and reduce the problem to the previous task.
2. Iterate within each subgroup over consecutive pairs of elements and pass them to `union`. By transitivity they will still "find" each other.

The second approach is more efficient.

In [ ]:
from typing import List, Tuple, Dict
from itertools import combinations

def extend_matches(groups: List[tuple]) -> List[tuple]:
    """Match predictions via transitive relation"""
    parent: Dict[int, int] = {}

    def find(x: int) -> int:
        """Find a parent for the input number"""
        if x not in parent:
            parent[x] = x
        while parent[x] != x:
            x = parent[x]
        return x

    def union(x: int, y: int) -> None:
        """Unite two numbers into one subset"""
        parent[find(x)] = find(y)

    for g in groups:
        """Iterate over subsets and merge two consequent numbers into one set"""
        for i in range(len(g) - 1):
            a, b = g[i], g[i+1]
            union(a, b)

    #Gather all groups
    all_groups: Dict[int, List[int]] = {}
    for p in parent.items():
        root = find(p[1])
        all_groups.setdefault(root, []).append(p[0])

    answer: List[tuple] = []
    for g in all_groups.items():
        answer.append(tuple(sorted(g[1])))
    
    return sorted(answer)
